In [1]:
import os, json
import curl_cffi.requests as requests

ENDPOINT = "https://caching.graphql.imdb.com/"
HEADERS = {
    "content-type": "application/json",
    "referer": "https://www.imdb.com/",
    "origin": "https://www.imdb.com",
    "x-imdb-client-name": "imdb-web-next-localized",
}
# The persisted-query hash rotates when IMDb rebuilds its JS bundle. Paste a fresh
# one from DevTools if you get "PersistedQueryNotFound" (or read it from env).
SHA256 = os.environ.get("IMDB_STORYLINE_SHA256", "bbf29ee4ceeefcf2d0825e0e57e3821aa2e11166b7cf820e1b40fb21095d7b08")

RATINGS_QUERY = ("query TitleRatingsHistogram($id: ID!) { title(id: $id) { "
                 "ratingsSummary { aggregateRating voteCount } "
                 "aggregateRatingsBreakdown { histogram { histogramValues { rating voteCount } } } } }")

IMDB_ID = "tt0903747"  # Breaking Bad

In [2]:
# Two operations in one JSON array -> one POST. Reply is array-ordered.
ops = [
    {"operationName": "Title_Storyline",
     "variables": {"isInMachineTranslateWeblab": True, "locale": "en-US", "titleId": IMDB_ID},
     "extensions": {"persistedQuery": {"sha256Hash": SHA256, "version": 1}}},
    {"operationName": "TitleRatingsHistogram", "query": RATINGS_QUERY, "variables": {"id": IMDB_ID}},
]

session = requests.Session(impersonate="chrome")
resp = session.post(ENDPOINT, headers=HEADERS, data=json.dumps(ops), timeout=25)
resp.raise_for_status()
storyline_json, histogram_json = resp.json()
print("HTTP", resp.status_code, "| 2 ops returned")

HTTP 200 | 2 ops returned


In [3]:
title = storyline_json["data"]["title"]
print("top-level keys:", list(title.keys()))

# How is a single plot nested? (this shape is why the parse below looks the way it does)
print(json.dumps(title.get("outlines"), indent=2)[:500])

top-level keys: ['id', 'summaries', 'outlines', 'synopses', 'storylineKeywords', 'taglines', 'genres', 'certificate', 'parentsGuide']
{
  "edges": [
    {
      "node": {
        "plotText": {
          "plaidHtml": "A chemistry teacher diagnosed with inoperable lung cancer turns to manufacturing and selling methamphetamine with a former student to secure his family&#39;s future."
        },
        "machineTranslatedText": {
          "language": {
            "id": "en-US"
          },
          "value": {
            "plaidHtml": "A chemistry teacher diagnosed with inoperable lung cancer turns to manufacturing and selling m


In [4]:
import re

def first_plot(node):
    edges = (node or {}).get("edges", [])
    if not edges:
        return None
    html = edges[0]["node"]["plotText"]["plaidHtml"]
    return re.sub(r"<[^>]+>", "", html).strip()          # crude tag-strip, just for display

print("outline :", first_plot(title.get("outlines")))
print("genres  :", [g["text"] for g in title.get("genres", {}).get("genres", [])])
print("keywords:", [e["node"]["text"] for e in title.get("storylineKeywords", {}).get("edges", [])][:8], "...")
print("tagline :", [e["node"]["text"] for e in title.get("taglines", {}).get("edges", [])][:1])
print("\nsummary :", (first_plot(title.get("summaries")) or "")[:300], "...")

outline : A chemistry teacher diagnosed with inoperable lung cancer turns to manufacturing and selling methamphetamine with a former student to secure his family&#39;s future.
genres  : ['Crime', 'Drama', 'Thriller']
keywords: ['cancer', 'chemistry', 'drug trade', 'methamphetamine', 'chemistry teacher'] ...
tagline : ['In the no-holds-barred world of Walt White, the end justifies the extreme. (season 2)']

summary : Walter White is a chemistry genius but works as a chemistry teacher at a high school in Albuquerque, New Mexico. His life drastically changes when he&#39;s diagnosed with stage III terminal lung cancer and given a prognosis of two years to live. To ensure that his pregnant wife and disabled teenage  ...


In [5]:
rt = histogram_json["data"]["title"]
print("ratingsSummary:", rt["ratingsSummary"])          # aggregate + total votes

buckets = rt["aggregateRatingsBreakdown"]["histogram"]["histogramValues"]
votes = {b["rating"]: b["voteCount"] for b in buckets}
for star in range(10, 0, -1):
    print(f"{star:>2} star: {votes.get(star, 0):,}")

ratingsSummary: {'aggregateRating': 9.5, 'voteCount': 2640901}
10 star: 1,759,719
 9 star: 479,259
 8 star: 183,861
 7 star: 68,057
 6 star: 26,745
 5 star: 15,295
 4 star: 8,208
 3 star: 6,759
 2 star: 12,443
 1 star: 80,555
